# Preprocesado

# 1. Carga y visualización de datos

Se cargan los datasets de entrenamiento (`train.csv`) y test (`test.csv`) utilizando pandas.

- `train` contiene la variable objetivo `Survived`
- `test` no contiene `Survived` (es lo que queremos predecir)

Trabajar con ambos es necesario para aplicar el mismo preprocesado posteriormente, por lo que vamos a juntarlos y posteriormente separarlos de nuevo.
Además, se van a crean copias de los datasets originales para evitar modificar los datos originales directamente.

Esto permite:
- Mantener los datos crudos intactos
- Repetir el preprocesado si es necesario
- Evitar errores difíciles de rastrear

In [4]:
import pandas as pd
import numpy as np

train = pd.read_csv("titanic_datasets/unprocessed/train.csv")
test = pd.read_csv("titanic_datasets/unprocessed/test.csv")

In [5]:
train_df = train.copy()
test_df = test.copy()

In [6]:
full = pd.concat([train_df, test_df], axis=0, ignore_index=True)

In [7]:
df = full.copy()

In [8]:
full.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
full.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Survived     891 non-null    float64
 2   Pclass       1309 non-null   int64  
 3   Name         1309 non-null   str    
 4   Sex          1309 non-null   str    
 5   Age          1046 non-null   float64
 6   SibSp        1309 non-null   int64  
 7   Parch        1309 non-null   int64  
 8   Ticket       1309 non-null   str    
 9   Fare         1308 non-null   float64
 10  Cabin        295 non-null    str    
 11  Embarked     1307 non-null   str    
dtypes: float64(3), int64(4), str(5)
memory usage: 122.8 KB


# 2. Feature Engineering

En esta fase se crean nuevas variables (features) a partir de las existentes.

Objetivo:
- Extraer información relevante no explícita
- Mejorar la capacidad predictiva del modelo

## Title

Vamos a extraer el título de cada pasajero a partir del nombre.

Ejemplo:
- "Braund, Mr. Owen Harris" → "Mr"

El título aporta información relevante sobre:
- Sexo
- Edad aproximada
- Estatus social

También se agrupan títulos poco frecuentes en la categoría "Rare" para evitar ruido en el modelo.

Además, vamos a normalizar títulos equivalentes de diferentes idiomas:
- Mlle, Ms → Miss
- Mme → Mrs

In [10]:
# 1. Creamos la columna extrayendo los títulos del nombre
df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
print("Antes de agrupar los títulos:")
print(df["Title"].value_counts())
print("-" * 30)

# 2. Hacemos los reemplazos de los títulos raros
df["Title"] = df["Title"].replace([
    "Lady", "Countess", "Capt", "Col", "Don", "Dr",
    "Major", "Rev", "Sir", "Jonkheer", "Dona"
], "Rare")

# 3. Normalizamos los títulos equivalentes
df["Title"] = df["Title"].replace({
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs"
})

print("Después de agrupar los títulos:")
print(df["Title"].value_counts())

Antes de agrupar los títulos:
Title
Mr          757
Miss        260
Mrs         197
Master       61
Rev           8
Dr            8
Col           4
Ms            2
Major         2
Mlle          2
Don           1
Mme           1
Lady          1
Sir           1
Capt          1
Countess      1
Jonkheer      1
Dona          1
Name: count, dtype: int64
------------------------------
Después de agrupar los títulos:
Title
Mr        757
Miss      264
Mrs       198
Master     61
Rare       29
Name: count, dtype: int64


## Tamaño de familia

Se crea la variable `FamilySize`:

FamilySize = SibSp + Parch + 1

Esto representa el número total de familiares a bordo (incluyendo al propio pasajero).

In [11]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

Además del tamaño exacto de familia, vamos a crear una categorización de si la familia es pequeña, mediana o grande.

In [12]:
def categorize_family(size):
    if size == 1:
        return 'Alone'
    elif 2 <= size <= 4:
        return 'Small'
    else:
        return 'Large'

df['FamilyCategory'] = df['FamilySize'].apply(categorize_family)

## Deck (cubierta del barco)

Se extrae la primera letra de la variable `Cabin`, que indica la cubierta.

Ejemplo:
- "C85" → "C"

Dado que hay muchos valores nulos, se rellenan como:
- "U" (Unknown)

Queremos extraer este dato ya que la cubierta puede estar relacionada con:
- Clase social
- Ubicación en el barco

In [13]:
df["Deck"] = df["Cabin"].str[0]
df["Deck"] = df["Deck"].fillna("Unknown")

## Ticket Group

Se calcula el número de pasajeros que comparten el mismo ticket.

Esto permite identificar grupos de personas que viajaban juntas.

La variable `TicketGroup` representa:
- Tamaño del grupo asociado al ticket

Puede aportar información similar a `FamilySize`, pero capturando relaciones no familiares.

In [14]:
df["TicketGroup"] = df.groupby("Ticket")["Ticket"].transform("count")

## Age group

La variable Age es continua. En la regresión logística, una relación lineal pura asume que el cambio en la probabilidad de supervivencia es constante por cada año que pasa. Sin embargo, tener 5 años frente a 15 años marcaba una diferencia abismal, pero tener 35 frente a 45 probablemente no. Por eso, vamos a agrupar las edades en categorías.

Antes de crear las categorías, como en la columna Age hay muchos valores nulos y no los queremos arrastrar a esta nueva varible, vamos a tratar esos valores.

In [15]:
# 1. Estudio inicial de la variable Age
print("--- 1. ANTES DE LA IMPUTACIÓN ---")
nulos_age_antes = df["Age"].isnull().sum()
porcentaje_nulos = (nulos_age_antes / len(df)) * 100
print(f"Nulos en Age: {nulos_age_antes} ({porcentaje_nulos:.2f}%)")
print(f"Media de edad: {df['Age'].mean():.2f} años")
print(f"Desviación estándar: {df['Age'].std():.2f}\n")

# 2. Análisis de la estrategia de imputación
print("--- 2. MEDIANAS CALCULADAS POR CLASE Y TÍTULO ---")
# Esto mostrará la tabla de valores que se van a usar para rellenar los huecos
tabla_medianas = df.groupby(["Pclass", "Title"])["Age"].median().unstack()
print(tabla_medianas)
print("\n" + "-"*40 + "\n")

# 3. Aplicación de la imputación
# Imputar Edad agrupando por Clase y Título
df["Age"] = df.groupby(["Pclass", "Title"])["Age"].transform(lambda x: x.fillna(x.median()))

# Red de seguridad por si alguna combinación Clase-Título no tenía ninguna edad registrada
df["Age"] = df["Age"].fillna(df["Age"].median())

# 4. Comprobación del impacto
print("--- 3. DESPUÉS DE LA IMPUTACIÓN ---")
nulos_age_despues = df["Age"].isnull().sum()
print(f"Nulos en Age restantes: {nulos_age_despues}")
print(f"Media de edad: {df['Age'].mean():.2f} años")
print(f"Desviación estándar: {df['Age'].std():.2f}")

# 5. Creación de los grupos de edad
bins = [0, 12, 18, 60, 120]
labels = ['Child', 'Teenager', 'Adult', 'Elderly']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)

print("--- 4. DISTRIBUCIÓN DE GRUPOS DE EDAD ---")
print(df['AgeGroup'].value_counts(dropna=False))

--- 1. ANTES DE LA IMPUTACIÓN ---
Nulos en Age: 263 (20.09%)
Media de edad: 29.88 años
Desviación estándar: 14.41

--- 2. MEDIANAS CALCULADAS POR CLASE Y TÍTULO ---
Title   Master  Miss    Mr   Mrs  Rare
Pclass                                
1          6.0  30.0  41.5  45.0  48.5
2          2.0  20.0  30.0  30.5  41.5
3          6.0  18.0  26.0  31.0   NaN

----------------------------------------

--- 3. DESPUÉS DE LA IMPUTACIÓN ---
Nulos en Age restantes: 0
Media de edad: 29.27 años
Desviación estándar: 13.45
--- 4. DISTRIBUCIÓN DE GRUPOS DE EDAD ---
AgeGroup
Adult       1027
Teenager     147
Child        102
Elderly       33
Name: count, dtype: int64


Podemos observar que la media y desviación estándar apenas han cambiado, por lo que la imputación ha sido un éxito. De este modo, los valores nulos no han sido arrastrados a la nueva variable.

## Verificación de variables

In [16]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Title,FamilySize,FamilyCategory,Deck,TicketGroup,AgeGroup
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Mr,2,Small,Unknown,1,Adult
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,Mrs,2,Small,C,2,Adult
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Miss,1,Alone,Unknown,1,Adult
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,Mrs,2,Small,C,2,Adult
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,Mr,1,Alone,Unknown,1,Adult


# 3. Gestión de valores nulos

In [17]:
# Calcular el total absoluto de nulos por columna
total_nulos = df.isnull().sum()

# Calcular el porcentaje que representan sobre el total del dataset
porcentaje_nulos = (df.isnull().sum() / len(df)) * 100

# Crear un DataFrame para mostrar los resultados
nulos_df = pd.DataFrame({
    'Total Nulos': total_nulos,
    'Porcentaje Nulos (%)': porcentaje_nulos
})
print(nulos_df)

                Total Nulos  Porcentaje Nulos (%)
PassengerId               0              0.000000
Survived                418             31.932773
Pclass                    0              0.000000
Name                      0              0.000000
Sex                       0              0.000000
Age                       0              0.000000
SibSp                     0              0.000000
Parch                     0              0.000000
Ticket                    0              0.000000
Fare                      1              0.076394
Cabin                  1014             77.463713
Embarked                  2              0.152788
Title                     0              0.000000
FamilySize                0              0.000000
FamilyCategory            0              0.000000
Deck                      0              0.000000
TicketGroup               0              0.000000
AgeGroup                  0              0.000000


Como podemos observar, los valores nulos se concentran en las variables Survived, Fare, Embarked y Cabin.

- Survived: No debemos modificar estos nulos, ya que corresponden a los pasajeros del dataset de test (los datos que queremos predecir). Al juntar ambos datasets para aplicar el preprocesado de forma homogénea se han visibilizado estos vacíos, pero los utilizaremos más adelante para volver a separar los conjuntos.

- Fare: Solo falta un dato. Al ser una variable numérica continua (tarifa), rellenarlo con la mediana es la opción más robusta a nivel estadístico, ya que nos asegura que el valor imputado no se verá sesgado por valores extremos (outliers), como los pasajeros de primera clase que pagaron tarifas excepcionalmente altas.

- Embarked: Solo falta la información de dos pasajeros. Al tratarse de una variable categórica (puertos C, Q, S), lo más lógico es asumir que subieron en el puerto con mayor afluencia. Por ello, imputaremos estos nulos utilizando la moda.

- Cabin: Falta casi el 80% de la información. Inventar o imputar una cantidad tan grande de datos introduciría un ruido que perjudicaría gravemente al modelo, por lo que la decisión más sensata es eliminar esta columna. Afortunadamente, en la etapa de Feature Engineering ya extrajimos el valor realmente útil de esta variable: la cubierta del barco, guardada en la variable Deck (rellenando los nulos con "Unknown"). El número exacto de camarote es un dato demasiado disperso y con demasiadas ausencias como para aportar valor predictivo.

Además de cabin, hay columnas que no influyen en nuestro estudio, como PassengerId, Name y Ticket, por lo que vamos a eliminalas también.

In [18]:
# 1. IMPUTACIÓN DE NULOS
# Fare: imputamos con la mediana
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Embarked: imputamos con el puerto más frecuente (la moda)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# 2. ELIMINACIÓN DE COLUMNAS INNECESARIAS
# Guardamos los IDs del test original
test_ids = df[df['Survived'].isnull()]['PassengerId'].copy()

# Borramos Cabin y las columnas de texto que ya no nos sirven
columnas_a_borrar = ["PassengerId", "Name", "Ticket", "Cabin"]
df = df.drop(columns=columnas_a_borrar)

# Comprobación asegurarnos de que estamos a 0 nulos (excluyendo Survived)
print("Nulos restantes (ignorando Survived):", df.drop(columns=['Survived']).isnull().sum().sum())

Nulos restantes (ignorando Survived): 0


# 4. One-Hot-Encoding

Vamos a transformar las columnas de texto en columnas de ceros y unos.

Teniendo en cuenta que es probable que usemos un modelo de regresión logística posteriormente, usaremos el parámetro `drop_first=True`. Esto nos ayuda a evitar la multicolinealidad entre variables. 
Por ejemplo, si creamos una columna Sex_male (1 si es hombre, 0 si no lo es), el modelo ya sabe implícitamente que si es 0, es mujer. No necesitamos una columna Sex_female, ya que darle información redundante confunde a la matemática de la Regresión Logística.

In [19]:
# Definimos cuáles son las columnas que contienen texto/categorías
categorical_cols = ["Sex", "Embarked", "Title", "Deck", "FamilyCategory", "AgeGroup"]

# Aplicamos get_dummies para convertirlas en 0s y 1s numéricos
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)

df.head()

,Survived,Pclass,Age,SibSp,Parch,Fare,FamilySize,TicketGroup,Sex_male,Embarked_Q,...,Deck_E,Deck_F,Deck_G,Deck_T,Deck_Unknown,FamilyCategory_Large,FamilyCategory_Small,AgeGroup_Teenager,AgeGroup_Adult,AgeGroup_Elderly
0,0.0,3,22.0,1,0,7.2500,2,1,1,0,...,0,0,0,0,1,0,1,0,1,0
1,1.0,1,38.0,1,0,71.2833,2,2,0,0,...,0,0,0,0,0,0,1,0,1,0
2,1.0,3,26.0,0,0,7.9250,1,1,0,0,...,0,0,0,0,1,0,0,0,1,0
3,1.0,1,35.0,1,0,53.1000,2,2,0,0,...,0,0,0,0,0,0,1,0,1,0
4,0.0,3,35.0,0,0,8.0500,1,1,1,0,...,0,0,0,0,1,0,0,0,1,0


# 5. Feature Scaling

A continuación, vamos a aplicar un escalado a nuestras variables numéricas utilizando `StandardScaler`. 

Este paso es recomendable porque el modelo que vamos a entrenar, la Regresión Logística, utiliza Descenso del Gradiente para aprender los pesos de cada variable. Si introducimos datos con escalas muy diferentes (por ejemplo, Fare con valores de cientos frente a Parch con valores de unidades), el descenso del gradiente se vuelve inestable, lento y le cuesta converger hacia la solución óptima. 

Al estandarizar todas las características numéricas para que tengan una media de 0 y una desviación estándar de 1, garantizamos que el algoritmo trabaje sobre un espacio equilibrado, converja rápidamente y no le dé una importancia falsa a una variable simplemente porque sus números son más grandes.

In [20]:
from sklearn.preprocessing import StandardScaler

# Definimos las variables que son números y tienen escalas diferentes
# (Omitimos Pclass porque ya representa un orden lógico: 1, 2, 3)
num_cols = ["Age", "Fare", "SibSp", "Parch", "FamilySize", "TicketGroup"]

# Inicializamos y aplicamos el escalador
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])